In [29]:
import pandas as pd
import yaml

In [30]:
def load_rules(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)["attacks"]
    
def convert_timestamps(df):
    # Zeek timestamps are Unix epoch (UTC)
    df["datetime"] = pd.to_datetime(df["ts"], unit="s", utc=True)
    # Convert to CICIDS2017 experiment timezone
    df["datetime"] = df["datetime"].dt.tz_convert("Canada/Atlantic")
    df["time"] = df["datetime"].dt.time
    return df

def label_flows_vectorized(df, rules):
    df["label"] = "BENIGN"

    for rule in rules:
        start = pd.to_datetime(rule["start"]).time()
        end = pd.to_datetime(rule["end"]).time()

        attacker_ips = rule["attacker_ips"]
        victim_ips = rule["victim_ips"]

        mask = (
            (df["time"] >= start) &
            (df["time"] <= end) &
            (
                (
                    df["id.orig_h"].isin(attacker_ips) &
                    df["id.resp_h"].isin(victim_ips)
                )
                |
                (
                    df["id.orig_h"].isin(victim_ips) &
                    df["id.resp_h"].isin(attacker_ips)
                )
            )
        )

        df.loc[mask, "label"] = rule["label"]

        print(rule["name"], "matches:", mask.sum())

    return df

In [31]:
df = pd.read_csv("zeek_logs/Wednesday-workingHours/conn.csv")
print("Loaded flows:", len(df))
df.head()

Loaded flows: 509391


,ts,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,duration,orig_bytes,resp_bytes,conn_state
0,1.499255e+09,192.168.10.14,49459,209.48.71.168,80,tcp,0.038308,0,0,SF
1,1.499255e+09,192.168.10.17,34492,192.168.10.3,53,udp,0.000224,46,196,SF
2,1.499255e+09,192.168.10.17,15221,192.168.10.3,53,udp,0.000190,46,196,SF
3,1.499255e+09,192.168.10.17,137,192.168.10.50,137,udp,0.000003,62,62,SF
4,1.499255e+09,192.168.10.17,3186,192.168.10.3,53,udp,0.000216,68,132,SF


In [32]:
df = convert_timestamps(df)

print("Dataset datetime range:")
print(df["datetime"].min())
print(df["datetime"].max())

print("\nDataset time range:")
print(df["time"].min(), "->", df["time"].max())

Dataset datetime range:
2017-07-05 08:42:42.084372044-03:00
2017-07-05 17:10:14.557249069-03:00

Dataset time range:
08:42:42.084372 -> 17:10:14.557249


In [33]:
df[(df["time"] > pd.to_datetime("09:45:00").time()) &
   (df["time"] < pd.to_datetime("10:15:00").time())]

,ts,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,duration,orig_bytes,resp_bytes,conn_state,datetime,time
39201,1.499259e+09,192.168.10.15,50468,50.63.243.230,80,tcp,0.069257,427,2266,SF,2017-07-05 09:45:02.639748096-03:00,09:45:02.639748
39205,1.499259e+09,192.168.10.15,50488,199.59.150.46,443,tcp,0.464732,1034,4835,SF,2017-07-05 09:45:05.881028891-03:00,09:45:05.881028
39206,1.499259e+09,192.168.10.15,50496,104.244.46.39,443,tcp,0.153165,806,15055,SF,2017-07-05 09:45:06.258954048-03:00,09:45:06.258954
39207,1.499259e+09,192.168.10.15,50497,104.244.46.39,443,tcp,0.171853,796,6803,SF,2017-07-05 09:45:06.263293981-03:00,09:45:06.263293
39208,1.499259e+09,192.168.10.15,50498,104.244.46.39,443,tcp,0.172120,795,6582,SF,2017-07-05 09:45:06.263400077-03:00,09:45:06.263400
...,...,...,...,...,...,...,...,...,...,...,...,...
75650,1.499260e+09,192.168.10.5,50662,216.58.219.205,443,tcp,14.335611,0,0,S1,2017-07-05 10:14:32.664864063-03:00,10:14:32.664864
75651,1.499260e+09,192.168.10.50,53708,192.168.10.3,3268,tcp,893.685236,34367,11731,RSTR,2017-07-05 10:04:48.498032093-03:00,10:04:48.498032
75833,1.499260e+09,192.168.10.50,54722,192.168.10.3,389,tcp,893.598525,35579,33329,RSTR,2017-07-05 10:05:20.586627007-03:00,10:05:20.586627
79486,1.499260e+09,192.168.10.12,37480,157.240.18.19,443,tcp,181.343455,1182,26148,SF,2017-07-05 10:14:43.147324085-03:00,10:14:43.147324


In [34]:
rules = load_rules("rules/wednesday.yaml")

In [36]:
df = label_flows_vectorized(df, rules)

DoS_Slowloris matches: 3864
DoS_Hulk matches: 154799
DoS_GoldenEye matches: 7739


In [37]:
df["label"].value_counts()

label
BENIGN           342989
DOS_HULK         154799
DOS_GOLDENEYE      7739
DOS_SLOWLORIS      3864
Name: count, dtype: int64

In [38]:
df.drop(columns=["datetime", "time"], inplace=True)

df.to_csv("wednesday_labeled.csv", index=False)

print("Saved labeled dataset: wednesday_labeled.csv")

Saved labeled dataset: wednesday_labeled.csv
